# Software Engineering 458 — Rat Detection Capstone

Sequential video classification with OpenCV, PyTorch, a frame encoder, and an LSTM.

**Data layout:** place this notebook in the project root and arrange videos like this:

```text
project/
  train/0/*.mp4
  train/1/*.mp4
  test/0/*.mp4
  test/1/*.mp4
```

Class folders `0` and `1` are the labels. Keep the data folders on both computers (or configure a shared location); Git does not automatically include local files that are ignored or too large to commit.

## Environment

Use the same Python environment and matching `torch` / `torchvision` versions on both computers. This notebook uses OpenCV to decode videos, avoiding the TorchCodec DLL loading failure saved in the earlier run. `d2l` is omitted because this notebook does not use it and its pinned NumPy dependency failed under Python 3.13.

In [ ]:
%pip install opencv-python

In [ ]:
from pathlib import Path
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_ROOT = Path.cwd()
TRAIN_DIR = PROJECT_ROOT / "train"
TEST_DIR = PROJECT_ROOT / "test"
NUM_FRAMES = 16
IMAGE_SIZE = 224
BATCH_SIZE = 4

print("Project folder:", PROJECT_ROOT)
print("Training folder:", TRAIN_DIR.resolve())
print("Test folder:", TEST_DIR.resolve())
print("PyTorch:", torch.__version__)

In [ ]:
def collect_videos(folder):
    """Return (paths, labels) for folder/{0,1}/video files."""
    folder = Path(folder)
    paths, labels = [], []
    if not folder.is_dir():
        return paths, labels

    for path in sorted(folder.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in {".mp4", ".mov", ".avi", ".mkv"}:
            continue
        relative_parts = path.relative_to(folder).parts
        if not relative_parts or relative_parts[0] not in {"0", "1"}:
            continue
        paths.append(path)
        labels.append(int(relative_parts[0]))
    return paths, labels

train_paths, train_labels = collect_videos(TRAIN_DIR)
test_paths, test_labels = collect_videos(TEST_DIR)

print(f"Training videos: {len(train_paths)}")
print(f"Test videos: {len(test_paths)}")
if not train_paths:
    raise FileNotFoundError(
        f"No training videos found under {TRAIN_DIR.resolve()}. "
        "Expected files under train/0 and/or train/1. Check the printed project folder."
    )
if len(train_paths) != len(train_labels) or len(test_paths) != len(test_labels):
    raise RuntimeError("A video path is missing a class label; check the 0/1 folder layout.")

In [ ]:
def find_motion(video_path, min_pixel_change=800, video_pad=2.0, frame_merge_gap=3.0):
    """Return padded (start, end) intervals with detected movement."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"OpenCV could not open video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    background = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=25)
    moving_timestamps = []
    frame_number = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        smaller = cv2.resize(frame, None, fx=0.5, fy=0.5)
        mask = background.apply(smaller)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, None, iterations=2)
        if cv2.countNonZero(mask) > min_pixel_change:
            moving_timestamps.append(frame_number / fps)
        frame_number += 1
    cap.release()

    windows = []
    for timestamp in moving_timestamps:
        if windows and timestamp - windows[-1][1] <= frame_merge_gap:
            windows[-1][1] = timestamp
        else:
            windows.append([timestamp, timestamp])
    return [(max(0, start - video_pad), end + video_pad) for start, end in windows]

# Optional example: find_motion(train_paths[0])


In [ ]:
class VideoDataset(Dataset):
    """Uniformly sample RGB frames and return [T, C, H, W] plus a binary label."""
    def __init__(self, video_paths, labels, num_frames=NUM_FRAMES, image_size=IMAGE_SIZE):
        if len(video_paths) != len(labels):
            raise ValueError("video_paths and labels must have the same length")
        self.video_paths = [Path(p) for p in video_paths]
        self.labels = [int(label) for label in labels]
        self.num_frames = num_frames
        self.image_size = image_size
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, index):
        path = self.video_paths[index]
        cap = cv2.VideoCapture(str(path))
        if not cap.isOpened():
            raise RuntimeError(f"OpenCV could not open video: {path}")

        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if frame_count <= 0:
            cap.release()
            raise RuntimeError(f"Video has no readable frames: {path}")

        frame_indices = np.linspace(0, frame_count - 1, self.num_frames).round().astype(int)
        frames = []
        for frame_index in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
            ok, frame = cap.read()
            if not ok:
                cap.release()
                raise RuntimeError(f"Could not read frame {frame_index} from {path}")
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)
            tensor = torch.from_numpy(frame.copy()).permute(2, 0, 1).float() / 255.0
            frames.append(tensor)
        cap.release()

        frames = torch.stack(frames)  # [T, C, H, W]
        frames = (frames - self.mean) / self.std
        label = torch.tensor(float(self.labels[index]), dtype=torch.float32)
        return frames, label

In [ ]:
train_dataset = VideoDataset(train_paths, train_labels)
test_dataset = VideoDataset(test_paths, test_labels)

# num_workers=0 is reliable in Jupyter on Windows and makes decode errors visible.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

sample_videos, sample_labels = next(iter(train_loader))
print("Video batch:", tuple(sample_videos.shape))  # [B, T, C, H, W]
print("Label batch:", tuple(sample_labels.shape))  # [B]

In [ ]:
classes = [0, 1]
train_counts = [sum(label == cls for label in train_labels) for cls in classes]
test_counts = [sum(label == cls for label in test_labels) for cls in classes]

x = np.arange(len(classes))
width = 0.35
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - width / 2, train_counts, width, label="Train")
ax.bar(x + width / 2, test_counts, width, label="Test")
ax.set(xticks=x, xticklabels=classes, xlabel="Class", ylabel="Videos", title="Videos per class")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Preview one sample after undoing ImageNet normalization.
frames, label = train_dataset[0]
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
indices = sorted(set([0, len(frames) // 2, len(frames) - 1]))
fig, axes = plt.subplots(1, len(indices), figsize=(4 * len(indices), 4))
for ax, i in zip(np.atleast_1d(axes), indices):
    frame = (frames[i] * std + mean).clamp(0, 1)
    ax.imshow(frame.permute(1, 2, 0))
    ax.set_title(f"Frame {i}; label {int(label)}")
    ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
class BinaryVideoClassifier(nn.Module):
    """Encode each frame with ResNet18, then classify the sequence with an LSTM."""
    def __init__(self, hidden_dim=256, dropout=0.3):
        super().__init__()
        backbone = resnet18(weights=None)
        feature_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.frame_encoder = backbone
        self.temporal_model = nn.LSTM(
            input_size=feature_dim,
            hidden_size=hidden_dim,
            batch_first=True,
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, videos):
        # videos: [B, T, C, H, W]
        batch_size, num_frames, channels, height, width = videos.shape
        frames = videos.reshape(batch_size * num_frames, channels, height, width)
        features = self.frame_encoder(frames)
        features = features.reshape(batch_size, num_frames, -1)
        sequence, _ = self.temporal_model(features)
        return self.classifier(sequence.mean(dim=1)).squeeze(1)  # [B]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BinaryVideoClassifier().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
print("Using device:", device)

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, total = 0.0, 0, 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for videos, labels in loader:
            videos = videos.to(device)
            labels = labels.to(device)
            logits = model(videos)
            loss = criterion(logits, labels)

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            correct += ((logits >= 0) == (labels >= 0.5)).sum().item()
            total += batch_size

    return total_loss / total, correct / total

In [ ]:
EPOCHS = 5
for epoch in range(EPOCHS):
    train_loss, train_accuracy = run_epoch(model, train_loader, criterion, optimizer)
    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"train loss {train_loss:.4f} | train accuracy {train_accuracy:.3f}"
    )

if len(test_dataset):
    test_loss, test_accuracy = run_epoch(model, test_loader, criterion)
    print(f"Test loss {test_loss:.4f} | test accuracy {test_accuracy:.3f}")
else:
    print("No test videos found; skipping evaluation.")

In [ ]:
model_path = PROJECT_ROOT / "binary_video_classifier.pt"
torch.save(model.state_dict(), model_path)
print("Saved model to:", model_path)